<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/forces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install m3gnet alignn

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/alignn/main/alignn/examples/sample_data_ff/mlearn_data/Si/id_prop.json

In [ ]:
from jarvis.db.jsonutils import loadjson
d = loadjson('id_prop.json')

In [ ]:

from alignn.ff.ff import default_path,ForceField
from jarvis.core.atoms import Atoms
model_path = default_path()
def get_alignn_forces(atoms=[]):
    
    ff = ForceField(
        jarvis_atoms=atoms,
        model_path=model_path,
    )
    en, fs = ff.unrelaxed_atoms()
    return fs

In [ ]:
from m3gnet.models import M3GNet, M3GNetCalculator, Potential
potential = Potential(M3GNet.load())
calculator = M3GNetCalculator(potential=potential, stress_weight=0.01)

def get_m3gnet_forces(atoms):
    atoms=atoms.ase_converter()
    atoms.calc = M3GNetCalculator(potential=potential)
    forces = atoms.get_forces()
    energy = atoms.get_potential_energy()
    return forces



In [ ]:
from tqdm import tqdm
mg=[]
al = []
dat=[]
for ii,i in tqdm(enumerate(d)):
  atoms = Atoms.from_dict(i['atoms'])
  mg_f = get_m3gnet_forces(atoms)
  mg.append(np.array(mg_f).flatten())
  al_f = get_alignn_forces(atoms)
  al.append(np.array(al_f).flatten())
  dat.append(np.array(i['forces']).flatten())

In [ ]:
len(dat)

In [ ]:
x = np.array(al).flatten()
y = x**2

In [ ]:
x

In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.array(dat).flatten(),np.array(al).flatten())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.plot(np.array(dat).flatten(),-1*x,'.')

In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.array(dat).flatten(),np.array(mg[0:14]).flatten())

In [ ]:
len(mg)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.plot(np.array(dat).flatten(),np.array(mg[0:14]).flatten(),'.')

In [ ]:
atoms.num_atoms